# XAUUSD XGBoost Training — Full Pipeline

This notebook exports 5 years of training data, validates it, trains a calibrated XGBoost model with macro features (US10Y + real DXY), and downloads the artifact for local deployment.

## Prerequisites
- Twelve Data API key (free tier works for 5 years)
- No GPU needed — runs on CPU in ~30-60 min for 50 Optuna trials

## Two paths:
1. **Auto-export**: Uses Twelve Data API to download everything fresh
2. **Upload bundle**: Upload a zip of `data/training/` exported from your local machine

In [ ]:
# @title 1. Mount Drive & Clone Repo
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or pull latest
import os
if os.path.exists('/content/tradebot'):
    %cd /content/tradebot
    !git pull
else:
    %cd /content
    !git clone https://github.com/<YOUR_USERNAME>/tradebot.git
    %cd tradebot

In [ ]:
# @title 2. Install Dependencies
# Install core + training extras
!pip install -e ".[train]"

In [ ]:
# @title 3. Set API Keys
# Option A: Use Colab Secrets (recommended)
from google.colab import userdata
import os

try:
    os.environ["TWELVE_DATA_API_KEY"] = userdata.get("TWELVE_DATA_API_KEY")
    print("TWELVE_DATA_API_KEY loaded from Colab Secrets")
except userdata.SecretNotFoundError:
    # Option B: Manual input
    TWELVE_DATA_API_KEY = input("Enter your Twelve Data API key: ")
    os.environ["TWELVE_DATA_API_KEY"] = TWELVE_DATA_API_KEY

# Verify
assert os.getenv("TWELVE_DATA_API_KEY"), "TWELVE_DATA_API_KEY is required"
print(f"Key starts with: {os.environ['TWELVE_DATA_API_KEY'][:8]}...")

In [ ]:
# @title 4. Choose Data Source
data_source = "export"  # @param ["export", "upload"]

if data_source == "upload":
    from google.colab import files
    print("Upload your training_bundle.zip (contains xauusd_m15.csv, etc.)")
    uploaded = files.upload()
    !mkdir -p data/training
    !unzip -o training_bundle.zip -d data/training/
    print("Upload complete. Files:")
    !ls -lh data/training/

In [ ]:
# @title 5. Export Training Data (auto-export path)
if data_source == "export":
    # Export XAUUSD + DXY proxy (EUR/USD) at M15/H1/H4
    !python scripts/export_training_bundle.py --years 5 --output-dir data/training

    # Export US10Y yields (from CalcFi — no API key needed)
    !python scripts/export_fred_series.py --output data/training/us10y_daily.csv

    # Export real DXY (needs Twelve Data API — uses same key)
    if os.path.exists('data/training/dxy_m15.csv'):
        print('DXY data already exported')
    else:
        !python scripts/export_twelve_data_history.py --symbol DXY --interval 15min --years 5 --output data/training/dxy_m15.csv

    print('\nTraining data files:')
    !ls -lh data/training/

In [ ]:
# @title 6. Validate Raw CSVs
!python scripts/validate_training_data.py --csv data/training/xauusd_m15.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h1.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h4.csv
!python scripts/validate_training_data.py --csv data/training/eurusd_m15.csv

# Validate DXY and US10Y if they exist
import os
if os.path.exists('data/training/dxy_m15.csv'):
    !python scripts/validate_training_data.py --csv data/training/dxy_m15.csv
if os.path.exists('data/training/us10y_daily.csv'):
    !python scripts/validate_training_data.py --csv data/training/us10y_daily.csv

In [ ]:
# @title 7. Sanitize OHLC (fix high/low inversions)
import glob
for f in sorted(glob.glob('data/training/*.csv')):
    !python scripts/sanitize_training_data.py "$f"

In [ ]:
# @title 8. Train Model
# Configuration — edit these as needed
TRIALS = 50              # Optuna trials (more = better but slower)
TARGET_MODE = "close_return"  # close_return, first_touch, three_class, binary
LOOKAHEAD = 8            # Candles to look ahead for target labeling
ATR_THRESHOLD = 1.0      # ATR multiplier for target threshold
BINARY = True            # True = overlap_macro_trend artifact (BUY-focused)

# Check if macro data is available
import os
has_macro = os.path.exists('data/training/us10y_daily.csv') and os.path.exists('data/training/dxy_m15.csv')
print(f"Macro features available: {has_macro}")

if has_macro:
    !python scripts/train_xgboost.py \
      --m15 data/training/xauusd_m15.csv \
      --h1 data/training/xauusd_h1.csv \
      --h4 data/training/xauusd_h4.csv \
      --dxy data/training/eurusd_m15.csv \
      --us10y data/training/us10y_daily.csv \
      --real-dxy data/training/dxy_m15.csv \
      --output models/xgb_xauusd_v1.pkl \
      --trials {TRIALS} \
      --target-mode {TARGET_MODE} \
      --lookahead {LOOKAHEAD} \
      --atr-threshold {ATR_THRESHOLD} \
      --binary
else:
    !python scripts/train_xgboost.py \
      --m15 data/training/xauusd_m15.csv \
      --h1 data/training/xauusd_h1.csv \
      --h4 data/training/xauusd_h4.csv \
      --dxy data/training/eurusd_m15.csv \
      --output models/xgb_xauusd_v1.pkl \
      --trials {TRIALS} \
      --target-mode {TARGET_MODE}

In [ ]:
# @title 9. Inspect Model Artifact
import joblib
artifact = joblib.load('models/xgb_xauusd_v1.pkl')
if isinstance(artifact, dict):
    print(f"artifact_type: {artifact.get('artifact_type')}")
    print(f"feature_columns: {len(artifact.get('feature_columns', []))} columns")
    print(f"threshold: {artifact.get('threshold')}")
    print(f"enabled_sides: {artifact.get('enabled_sides')}")
    model = artifact.get('model')
    print(f"model type: {type(model).__name__}")
    if hasattr(model, 'classes_'):
        print(f"classes_: {model.classes_}")
else:
    print(f"Raw model: {type(artifact).__name__}")
    print(f"classes_: {artifact.classes_}")

In [ ]:
# @title 10. Download Artifact
from google.colab import files
files.download('models/xgb_xauusd_v1.pkl')

---
## Post-Training: Deploy Locally

1. Move the downloaded `xgb_xauusd_v1.pkl` into your local `models/` directory:
   ```bash
   mv ~/Downloads/xgb_xauusd_v1.pkl models/overlap_macro_trend_xgb.pkl
   ```

2. No config changes needed — `config.yaml` already points to `models/overlap_macro_trend_xgb.pkl`

3. Verify it loads:
   ```bash
   python -c "
   import joblib
   a = joblib.load('models/overlap_macro_trend_xgb.pkl')
   print('OK, artifact_type=' + a.get('artifact_type', 'raw'))
   "
   ```

4. Run a dry signal to test:
   ```bash
   python scripts/dry_model_signal.py --ignore-calendar
   ```

## Training with the Experiment Script (Advanced)

For walk-forward validation, candidate family labeling, and execution-aware training, use the experiment script:

```bash
python scripts/run_overlap_macro_trend_experiment.py \
  --m15 data/training/xauusd_m15.csv \
  --h1 data/training/xauusd_h1.csv \
  --h4 data/training/xauusd_h4.csv \
  --dxy data/training/dxy_m15.csv \
  --us10y data/training/us10y_daily.csv \
  --folds 5 --trials 25 \
  --export-if-enabled \
  --output models/overlap_macro_trend_xgb.pkl
```